<div style="background-color:white;padding:20px; color:black">
<h1 style= "background-color: #A72608; 
             text-align:left;
             color:white;
             font-weight:600;
             font-size:25px;
             border:0;
             border-left:solid 10px black;
             border-radius: 0px;
             padding: 20px 20px;"> 
 Introduction to Graph Neural Networks (GNNs)
 </h1>
</div>

<div style="background-color:white;padding:20px; color:black">

<h1 style= "background-color: #ffffff; 
             text-align:left;
             font-weight:600;
             color:crimson;
             font-size:25px;
             border-bottom:solid 2px gray;
             border-radius: 0px;
             padding: 0px 0px;"> 
Section I. 
</h1>

<h2 style= "background-color: #ffffff; 
             text-align:left;
             font-weight:600;
             color:#D2691E;
             font-size:20px;
             border-radius: 0px;
             padding: 0px 10px;"> 
From SMILES to Graphs: Converting Molecular Representations 
</h2>

It is important that we are able to convert one molecular representation into another. So our first task is: 

- Conversion of a SMILES string to a graph.

</div>




In [1]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
# import torch
# import torch.nn as nn
# import torch.nn.functional as F
import pandas as pd
import rdkit, rdkit.Chem, rdkit.Chem.rdDepictor, rdkit.Chem.Draw
# import networkx as nx


#For ignoring/blocking warnings
import warnings
warnings.filterwarnings('ignore')

from rdkit import rdBase
blocker = rdBase.BlockLogs()

<div style="color:black; background-color: #f0f8ff; padding: 5px; border-radius: 5px;">

### 🔵 Solubility Dataset

In the code below, we load a solubility dataset and convert it from 'SMILES' strings into `rdkit` `Mol` objects. For that we can use `rdkit`'s method `MolFromSmiles` which is in the `Chem` module. 

</div>

In [2]:
soldata = pd.read_csv(
    "https://github.com/whitead/dmol-book/raw/main/data/curated-solubility-dataset.csv"
)
# filter any whose smiles don't read in correctly (this changes depending on rdkit version, so we check here)
N = len(soldata)
soldata = soldata[soldata.SMILES.apply(lambda x: rdkit.Chem.MolFromSmiles(x) is not None)].reset_index(drop=True)
print(f"Dropped {N - len(soldata)} rows with invalid SMILES")

np.random.seed(0)


Dropped 2 rows with invalid SMILES


<div style="background-color:white;padding:20px; color:black">

We can go from:

SMILES -> rdkit `Mol`. Can we go one step further and convert `Mol` to Graphs? If so then we can:

SMILES -> Graph

But for that we need to create:

+ A node matrix (tells us which node is which element)
+ An adjacency matrix (which edge connects which nodes)

The `smiles2graph` function given below creates:

+ a one-hot node feature vectors for the element C, H, and O. 
+ It also creates an adjacency tensor with one-hot bond order being the feature vector.

</div>




In [11]:
def smiles2graph(sml):
    """Argument for the RD2NX function should be a valid SMILES sequence
    returns: the graph
    """
    m = rdkit.Chem.MolFromSmiles(sml)
    m = rdkit.Chem.AddHs(m)
    order_string = {
        rdkit.Chem.rdchem.BondType.SINGLE: 1,
        rdkit.Chem.rdchem.BondType.DOUBLE: 2,
        rdkit.Chem.rdchem.BondType.TRIPLE: 3,
        rdkit.Chem.rdchem.BondType.AROMATIC: 4,
    }
    N = len(list(m.GetAtoms()))
    nodes = np.zeros((N, len(my_elements)))
    lookup = list(my_elements.keys())
    for i in m.GetAtoms():
        nodes[i.GetIdx(), lookup.index(i.GetAtomicNum())] = 1

    adj = np.zeros((N, N, 5))
    for j in m.GetBonds():
        u = min(j.GetBeginAtomIdx(), j.GetEndAtomIdx())
        v = max(j.GetBeginAtomIdx(), j.GetEndAtomIdx())
        order = j.GetBondType()
        if order in order_string:
            order = order_string[order]
        else:
            raise Warning("Ignoring bond order" + order)
        adj[u, v, order] = 1
        adj[v, u, order] = 1
    return nodes, adj

nodes, adj = smiles2graph("CO")
nodes

nodes_df = pd.DataFrame(nodes, columns = ['C', 'O', 'H'])
display(nodes_df)

,C,O,H
0,1.0,0.0,0.0
1,0.0,1.0,0.0
2,0.0,0.0,1.0
3,0.0,0.0,1.0
4,0.0,0.0,1.0
5,0.0,0.0,1.0


In [10]:
adj

array([[[0., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0.],
        [0., 1., 0., 0., 0.],
        [0., 1., 0., 0., 0.],
        [0., 1., 0., 0., 0.],
        [0., 0., 0., 0., 0.]],

       [[0., 1., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0.]],

       [[0., 1., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]],

       [[0., 1., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]],

       [[0., 1., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]],

       [[0., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0.],
        [0., 0., 0., 0., 0.],


<div style="background-color:#F0E68C;padding:5px; color:black">

<h2 style= "text-align:left;
             color:#556B2F;
             font-weight:bold;
             font-size:25px;
             border-radius: 0px;
             padding: 5px 10px;"> 
Task</h2>

Create adjacency matrices for the following molecules:

1. Propane
2. Ethene (look online if you can't find it)
3. Cyclohexane
4. O=C(C)Oc1ccccc1C(=O)O
5. CN1C=NC2=C1C(=O)N(C(=O)N2C)C
6. A molecule of your choice.

<!-- Can you identify what are molecules 4 and 5? -->

<!-- <details name="my-tabs" style="color:#800000">
  <summary>Hint: Propane's chemical formule is C₃H₈ so its SMILES would be:</summary>
  <div class="tab-content" style="font-weight:bold;color:green;padding:0em 1em">
    <p>CCC.</p>
  </div>
</details> -->
</div>

<div style="background-color:white;padding-left:10px;color:black">

With that done we can move to our next task. We need a mechanism which:

- Allows gathering of information. For that we will create what is called a **Graph Convolution Layer (GCN)**

GCNs will be integral the GNN model we will develop, so lets remind ourselves what a GNN is.



<h1 style= "background-color: #ffffff; 
             text-align:left;
             font-weight:600;
             color:crimson;
             font-size:25px;
             border-bottom:solid 2px gray;
             border-radius: 0px;
             padding: 0px 0px;"> 
Section II. 
</h1>

<h2 style= "background-color: #ffffff; 
             text-align:left;
             font-weight:600;
             color:#D2691E;
             font-size:20px;
             border-radius: 0px;
             padding: 0px 10px;"> 
Graph Neural Networks
</h2>

A graph neural network (GNN) is a neural network with two defining attributes:

1. Its input is a graph
1. Its output is permutation equivariant

We can understand clearly the first point. Here, a graph permutation means re-ordering our nodes. In our methanol example above, we could have easily made the carbon be atom 1 instead of atom 4. Our new adjacency matrix would then be:

A GNN is permutation equivariant if the output change the same way as these exchanges. If you are trying to model a per-atom quantity like partial charge or chemical shift, this is obviously essential. If you change the order of atoms input, you would expect the order of their partial charges to similarly change.
</div>




<div style="background-color:white;padding:20px; color:black">

<h2 style= "background-color: #ffffff; 
             text-align:left;
             font-weight:600;
             color:#D2691E;
             font-size:20px;
             border-radius: 0px;
             padding: 0px 10px;"> 
A simple GNN
</h2>

We will often mention a GNN when we really mean a **layer** from a GNN. Most GNNs implement a specific layer that can deal with graphs, and so usually we are only concerned with this layer. Let’s see an example of a simple layer for a GNN:

$$
\begin{equation}
f_k = \sigma\left( \sum_i \sum_j v_{ij}w_{jk}  \right)
\end{equation}
$$

This equation shows that we first multiply every node (
) feature by trainable weights
, sum over all node features, and then apply an activation. This will yield a single feature vector for the graph. Is this equation permutation invariant? Yes, because the node index in our expression is index
which can be re-ordered without affecting the output.

</div>




<div style="background-color:white;padding:10px; color:black">

<h1 style= "background-color: #ffffff; 
             text-align:left;
             font-weight:600;
             color:crimson;
             font-size:25px;
             border-bottom:solid 2px gray;
             border-radius: 0px;
             padding: 0px 0px;"> 
Section III. 
</h1>

<h2 style= "background-color: #ffffff; 
             text-align:left;
             font-weight:600;
             color:#D2691E;
             font-size:20px;
             border-radius: 0px;
             padding: 0px 0px;"> 
Graph Convolutions (Kipf and Welling)
</h2>

GCNs are important for understanding other GNNs. A GCN layer is a way to “communicate” between a node and its neighbors. 

The output for node $i$ will depend only on its *immediate* neighbors. 

For chemistry, this is not satisfactory. You can stack multiple layers though. If you have two layers, the output for node
will include information about node’s neighbors’ neighbors. 

Another important detail to understand in GCNs is that the averaging procedure accomplishes two goals: 

1. it gives permutation equivariance by removing the effect of neighbor order and 
1. it prevents a change in magnitude in node features. A sum would accomplish (i) but would cause the magnitude of the node features to grow after each layer. Of course, you could ad-hoc put a batch normalization layer after each GCN layer to keep output magnitudes stable but averaging is easy.
</div>




<div style="background-color:white;padding:10px; color:black">

<h2 style= "background-color: #ffffff; 
             text-align:left;
             font-weight:600;
             color:#D2691E;
             font-size:20px;
             border-radius: 0px;
             padding: 0px 0px;"> 
Code Implementation  (as a layer)
</h2>

We now create a tensor implementation of the GCN. Let's skip the activation and trainable weights for now and first compute our rank 2 adjacency matrix. 

The `smiles2graph` code we just used computes an adjacency tensor with feature vectors. We can fix that with a simple reduction and add the identity at the same time
</div>




In [6]:
nodes, adj = smiles2graph("CO")
adj_mat = np.sum(adj, axis=-1) + np.eye(adj.shape[0])
adj_mat

array([[1., 1., 1., 1., 1., 0.],
       [1., 1., 0., 0., 0., 1.],
       [1., 0., 1., 0., 0., 0.],
       [1., 0., 0., 1., 0., 0.],
       [1., 0., 0., 0., 1., 0.],
       [0., 1., 0., 0., 0., 1.]])

In [7]:
degree = np.sum(adj_mat, axis=-1)
degree

array([5., 3., 2., 2., 2., 2.])

In [8]:
print(nodes[0])
# note to divide by degree, make the input 1 / degree
new_nodes = np.einsum("i,ij,jk->ik", 1 / degree, adj_mat, nodes)
print(new_nodes[0])

[1. 0. 0.]
[0.2 0.2 0.6]


<div style="background-color:white;padding:20px; color:black">

To now implement this as a *layer* in PyTorch, we must put this code above into a new Module subclass. 

The three main changes are that we:

- create trainable parameters self.w and use them in the torch.einsum, 
- we use an activation, 
- we output both our new node features and the adjacency matrix. 

The reason to output the adjacency matrix is so that we can stack multiple GCN layers without having to pass the adjacency matrix each time.
</div>




In [ ]:
class GCNLayer(nn.Module):
    """Implementation of GCN as layer"""

    def __init__(self, in_features, out_features, activation=None):
        super(GCNLayer, self).__init__()
        self.w = nn.Parameter(torch.empty(in_features, out_features))
        nn.init.xavier_uniform_(self.w)
        self.activation = activation

    def forward(self, nodes, adj):
        degree = torch.sum(adj, dim=-1)
        new_nodes = torch.einsum("bi,bij,bjk,kl->bil", 1 / degree, adj, nodes, self.w)
        if self.activation is not None:
            new_nodes = self.activation(new_nodes)
        return new_nodes, adj

<div style="background-color:white;padding:20px; color:black">

Note how our GCN layer outputs node-level features (just like our 'game' in the workshop yielded individual level results)

</div>




<div style="background-color:white;padding:20px; color:black">

<h1 style= "background-color: #ffffff; 
             text-align:left;
             font-weight:600;
             color:crimson;
             font-size:25px;
             border-bottom:solid 2px gray;
             border-radius: 0px;
             padding: 0px 0px;"> 
Section IV. 
</h1>

<h2 style= "background-color: #ffffff; 
             text-align:left;
             font-weight:600;
             color:#D2691E;
             font-size:20px;
             border-radius: 0px;
             padding: 0px 0px;"> 
Solubility Example
</h2>

We’ll now revisit predicting solubility with GCNs. 

Remember before that we used the features included with the dataset. Now we can use the molecular structures directly. 

Our GCN layer outputs node-level features. But solubility is a property of the molecule and to predict solubility, we need to get a graph-level feature. 

A crude way to do that is to just take the average over all node features after our GCN layers. This is simple, permutation invariant, and gets us from node-level to graph level. Here’s an implementation of this

</div>




In [9]:
class GRLayer(nn.Module):
    """A GNN layer that computes average over all node features"""

    def __init__(self):
        super(GRLayer, self).__init__()

    def forward(self, nodes, adj):
        reduction = torch.mean(nodes, dim=1) #The key line: just to compute the mean over the nodes (dim=1):
        return reduction

NameError: name 'nn' is not defined

<div style="background-color:white;padding:20px; color:black">

<h2 style= "background-color: #ffffff; 
             text-align:left;
             font-weight:600;
             color:#D2691E;
             font-size:20px;
             border-radius: 0px;
             padding: 0px 0px;"> 
The model
</h2>
Let's add some dense layers and make sure we have a single-output without activation since we’re doing regression.
</div>

In [ ]:
class SolubilityModel(nn.Module):
    def __init__(self, node_features=100):
        super(SolubilityModel, self).__init__()
        self.gcn1 = GCNLayer(node_features, node_features, activation=F.relu)
        self.gcn2 = GCNLayer(node_features, node_features, activation=F.relu)
        self.gcn3 = GCNLayer(node_features, node_features, activation=F.relu)
        self.gcn4 = GCNLayer(node_features, node_features, activation=F.relu)
        self.gr = GRLayer()
        self.fc1 = nn.Linear(node_features, 16)
        self.fc2 = nn.Linear(16, 1)
    
    def forward(self, nodes, adj):
        x, adj = self.gcn1(nodes, adj)
        x, adj = self.gcn2(x, adj)
        x, adj = self.gcn3(x, adj)
        x, adj = self.gcn4(x, adj)
        x = self.gr(x, adj)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = SolubilityModel()

<div style="background-color:white;padding:20px; color:black">
where does the 100 come from? Well, this dataset has lots of elements so we cannot use our size 3 one-hot encodings because we’ll have more than 3 unique elements. We previously only had C, H and O. This is a good time to update our smiles2graph function to deal with this.
</div>



In [ ]:
def gen_smiles2graph(sml):
    """Argument for the RD2NX function should be a valid SMILES sequence
    returns: the graph
    """
    m = rdkit.Chem.MolFromSmiles(sml)
    m = rdkit.Chem.AddHs(m)
    order_string = {
        rdkit.Chem.rdchem.BondType.SINGLE: 1,
        rdkit.Chem.rdchem.BondType.DOUBLE: 2,
        rdkit.Chem.rdchem.BondType.TRIPLE: 3,
        rdkit.Chem.rdchem.BondType.AROMATIC: 4,
    }
    N = len(list(m.GetAtoms()))
    nodes = np.zeros((N, 100))
    for i in m.GetAtoms():
        nodes[i.GetIdx(), i.GetAtomicNum()] = 1

    adj = np.zeros((N, N))
    for j in m.GetBonds():
        u = min(j.GetBeginAtomIdx(), j.GetEndAtomIdx())
        v = max(j.GetBeginAtomIdx(), j.GetEndAtomIdx())
        order = j.GetBondType()
        if order in order_string:
            order = order_string[order]
        else:
            raise Warning("Ignoring bond order" + order)
        adj[u, v] = 1
        adj[v, u] = 1
    adj += np.eye(N)
    return nodes, adj

In [ ]:
nodes, adj = gen_smiles2graph("CO")
nodes_t = torch.from_numpy(nodes[np.newaxis, ...]).float()
adj_mat_t = torch.from_numpy(adj[np.newaxis, ...]).float()
model(nodes_t, adj_mat_t)

It outputs one number! That’s always nice to have. Now we need to do some work to get a trainable dataset. Our dataset is a little bit complex because our features are tuples of tensors(V,E) so that our dataset is a tuple of tuples: ((V,E),y). We use a PyTorch Dataset class, which requires implementing __len__ and __getitem__ methods.

In [4]:
from torch.utils.data import Dataset, DataLoader

class SolubilityDataset(Dataset):
    def __init__(self, data, start_idx=0, end_idx=None):
        self.data = data
        self.start_idx = start_idx
        self.end_idx = end_idx if end_idx is not None else len(data)
    
    def __len__(self):
        return self.end_idx - self.start_idx
    
    def __getitem__(self, idx):
        idx = idx + self.start_idx
        nodes, adj = gen_smiles2graph(self.data.SMILES[idx])
        sol = self.data.Solubility[idx]
        # this time we do not batch - it will be done by dataloader
        nodes_t = torch.from_numpy(nodes).float()
        adj_t = torch.from_numpy(adj).float()
        return (nodes_t, adj_t), torch.tensor(sol, dtype=torch.float32)

Now we can split our dataset into train, validation, and test sets. I just used 200/200 for testing and validation here.

In [ ]:
test_data = SolubilityDataset(soldata, 0, 200)
val_data = SolubilityDataset(soldata, 200, 400)
train_data = SolubilityDataset(soldata, 400)

And finally, time to train.

In [ ]:
train_loader = DataLoader(train_data, batch_size=1, shuffle=True)
val_loader = DataLoader(val_data, batch_size=1, shuffle=False)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

history = {"loss": [], "val_loss": []}
epochs = 10

for epoch in range(epochs):
    model.train()
    train_loss = 0
    for (nodes, adj), y in train_loader:
        optimizer.zero_grad()
        y_pred = model(nodes, adj)
        loss = criterion(y_pred.squeeze(), y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for (nodes, adj), y in val_loader:
            y_pred = model(nodes, adj)
            loss = criterion(y_pred.squeeze(), y)
            val_loss += loss.item()
    
    train_loss /= len(train_loader)
    val_loss /= len(val_loader)
    history["loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    print(f"Epoch {epoch+1}/{epochs}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

In [ ]:
plt.plot(history["loss"], label="training")
plt.plot(history["val_loss"], label="validation")
plt.legend()
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()

This model is definitely underfit. One reason is that our batch size is 1. This is a side-effect of making the number of atoms variable, because otherwise we have trouble batching together our data if there are two unknown dimensions. A standard trick is to group together multiple molecules into one graph, but making sure they are disconnected (no bonds between the molecules). That allows you to batch molecules without increasing the rank of your model/data.

Let’s now check the parity plot.

In [ ]:
model.eval()
yhat = []
test_y = []
test_loader = DataLoader(test_data, batch_size=1, shuffle=False)
with torch.no_grad():
    for (nodes, adj), y in test_loader:
        pred = model(nodes, adj)
        yhat.append(pred.squeeze().item())
        test_y.append(y.item())

plt.figure()
plt.plot(test_y, test_y, "-")
plt.plot(test_y, yhat, ".")
plt.text(
    min(test_y) + 1,
    max(test_y) - 2,
    f"correlation = {np.corrcoef(test_y, yhat)[0,1]:.3f}",
)
plt.text(
    min(test_y) + 1,
    max(test_y) - 3,
    f"loss = {np.sqrt(np.mean((np.array(test_y) - np.array(yhat))**2)):.3f}",
)
plt.title("Testing Data")
plt.show()

## Sources

The principal source for this notebook is:

https://dmol.pub/dl/gnn.html#a-graph-neural-network